# 06 · Running External Programs: `subprocess`, `shlex`, Signals & Environment

**Module 11 – Files & OS** · Estimated time: 45 min

> ❗ Not in either source (the handbook's `os` section doesn't cover it). Automation scripts constantly run other programs: `git`, `ffmpeg`, `docker`, compilers, shell utilities.

### 🎯 What you will learn
- ✅ `subprocess.run()`: capture output, check errors, timeouts, input
- Why `shell=True` is dangerous, and `shlex` for safe quoting
- Environment variables and working directories for child processes
- Streaming output and pipelines with `Popen`
- Running Python itself as a subprocess (`sys.executable`)
- Exit codes, signals and timeouts
- ⚠️ Old APIs: `os.system`, `os.popen`, `commands`

## 1. `subprocess.run`: the one function you need most

In [1]:
import subprocess, sys, shlex, os

result = subprocess.run(["git", "--version"], capture_output=True, text=True)
print("return code:", result.returncode)
print("stdout     :", result.stdout.strip())
print("stderr     :", repr(result.stderr))

return code: 0
stdout     : git version 2.50.1 (Apple Git-155)
stderr     : ''


| Argument | Meaning |
|---|---|
| `args` (a **list**) | program + arguments, no shell parsing |
| `capture_output=True` | collect stdout/stderr (otherwise they go to your terminal) |
| `text=True` | decode output to `str` (otherwise `bytes`); add `encoding="utf-8"` to be explicit |
| `check=True` | raise `CalledProcessError` if the exit code ≠ 0 |
| `timeout=seconds` | kill and raise `TimeoutExpired` if it takes too long |
| `input="..."` | send data to the program's stdin |
| `cwd="path"` | run in another directory |
| `env={...}` | environment variables for the child |

In [2]:
r = subprocess.run([sys.executable, "-c", "import sys; print(sys.stdin.read().upper())"],
                   input="hello from the parent", capture_output=True, text=True)
print(r.stdout)

HELLO FROM THE PARENT



### Handling failures

In [3]:
try:
    subprocess.run([sys.executable, "-c", "import sys; print('oops', file=sys.stderr); sys.exit(3)"],
                   capture_output=True, text=True, check=True)
except subprocess.CalledProcessError as e:
    print(f"exit code {e.returncode}; stderr: {e.stderr.strip()}")

try:
    subprocess.run([sys.executable, "-c", "import time; time.sleep(5)"], timeout=0.5)
except subprocess.TimeoutExpired as e:
    print("TimeoutExpired after", e.timeout, "s")

try:
    subprocess.run(["definitely-not-a-real-program"])
except FileNotFoundError as e:
    print("FileNotFoundError:", e.strerror)

exit code 3; stderr: oops


TimeoutExpired after 0.5 s
FileNotFoundError: No such file or directory


## 2. ⚠️ `shell=True` and command injection

With `shell=True` the command is a **string** interpreted by the shell (`/bin/sh` or `cmd.exe`). Convenient for pipes and wildcards, but **dangerous with user input**:

In [4]:
filename = "notes.txt; echo HACKED"                 # imagine this came from a user
r = subprocess.run(f"echo processing {filename}", shell=True, capture_output=True, text=True)
print(r.stdout)                                      # the injected command ran!

r = subprocess.run(["echo", "processing", filename], capture_output=True, text=True)
print(r.stdout)                                      # ✅ list form: the filename is ONE argument

processing notes.txt
HACKED

processing notes.txt; echo HACKED



If you truly need a shell string, quote untrusted parts with `shlex.quote`; to turn a command string into a safe list, use `shlex.split`:

In [5]:
print(shlex.quote(filename))
print(shlex.split('grep -n "hello world" "my file.txt"'))
print(shlex.join(["ls", "-l", "my file.txt"]))

'notes.txt; echo HACKED'
['grep', '-n', 'hello world', 'my file.txt']
ls -l 'my file.txt'


## 3. Environment and working directory

In [6]:
env = {**os.environ, "APP_MODE": "test", "PYTHONUTF8": "1"}        # extend, don't replace, the environment
r = subprocess.run([sys.executable, "-c", "import os; print(os.environ['APP_MODE'], os.getcwd())"],
                   env=env, cwd="/", capture_output=True, text=True)
print(r.stdout)

test /



> ⚠️ Passing `env={"APP_MODE": "test"}` alone **removes** `PATH` and everything else from the child's environment; always start from `os.environ`.

## 4. Running Python code & tools as subprocesses

Always use **`sys.executable`** (the current interpreter, venv-aware), not `"python"`:

In [7]:
for tool in (["-m", "pip", "--version"], ["-m", "json.tool", "--help"]):
    r = subprocess.run([sys.executable, *tool], capture_output=True, text=True)
    print(r.stdout.splitlines()[0] if r.stdout else r.stderr.splitlines()[0])

pip 26.2.1 from /private/tmp/claude-501/-Users-gaurvendrasingh-Desktop-python/f4ffaf20-c7ab-4c1c-ae64-d5055060b56d/scratchpad/venv315/lib/python3.15/site-packages/pip (python 3.15)
usage: python -m json.tool [-h] [--sort-keys] [--no-ensure-ascii]


## 5. Streaming output and pipelines with `Popen`

`run()` waits until the program finishes. `Popen` starts it and lets you interact while it runs:

In [8]:
code = "import time\nfor i in range(3):\n    print(f'step {i}', flush=True)\n    time.sleep(0.2)"
with subprocess.Popen([sys.executable, "-c", code], stdout=subprocess.PIPE, text=True) as proc:
    for line in proc.stdout:                 # read lines as they arrive (progress, logs)
        print("got:", line.strip())
print("exit code:", proc.returncode)

got: step 0


got: step 1


got: step 2


exit code: 0


In [9]:
# A pipeline: producer | consumer, without a shell
producer = subprocess.Popen([sys.executable, "-c", "print('\\n'.join(['banana','apple','cherry']))"],
                            stdout=subprocess.PIPE)
consumer = subprocess.Popen([sys.executable, "-c", "import sys; print(sorted(sys.stdin.read().split()))"],
                            stdin=producer.stdout, stdout=subprocess.PIPE, text=True)
producer.stdout.close()                     # let the producer get SIGPIPE if the consumer exits
print(consumer.communicate()[0])

['apple', 'banana', 'cherry']



| `Popen` method | Purpose |
|---|---|
| `communicate(input, timeout)` | send input, read all output, wait (avoids pipe deadlocks) |
| `poll()` | `None` while running, else the exit code |
| `wait(timeout)` | block until it exits |
| `terminate()` / `kill()` | ask politely (SIGTERM) / force (SIGKILL) |
| `send_signal(sig)` | any signal (POSIX) |

In [10]:
proc = subprocess.Popen([sys.executable, "-c", "import time; time.sleep(30)"])
print("running?", proc.poll() is None)
proc.terminate()
proc.wait(timeout=5)
print("exit code after terminate:", proc.returncode)     # negative on POSIX = killed by that signal

running? True
exit code after terminate: -15


## 6. ⚠️ Old ways you'll still see

| Old | Problem | Use |
|---|---|---|
| `os.system("cmd")` | shell injection, no output capture, only an exit status | `subprocess.run([...])` |
| `os.popen("cmd").read()` | shell, poor error handling | `subprocess.run(..., capture_output=True)` |
| `subprocess.call` / `check_call` / `check_output` | older helper functions (still fine) | `subprocess.run(..., check=True, capture_output=True)` |
| `commands` module | Python 2 | – |

> 💡 For async programs, use `asyncio.create_subprocess_exec` (Module 13 · 07). For complex shell scripting in Python, libraries like `sh` or `plumbum` exist, but plain `subprocess` is usually enough.

## 📝 Summary
- Use `subprocess.run([...], capture_output=True, text=True, check=True, timeout=...)`.
- Pass commands as **lists**; avoid `shell=True` with untrusted input (`shlex.quote`/`split` when needed).
- Extend `os.environ` for child environments; use `sys.executable` to run Python.
- `Popen` for streaming, pipelines and process control.

## 🏋️ Practice
1. Write `git_branch()` that returns the current Git branch name (or `None` if not in a repo).
2. Run `python -m pip list --format=json` and print the 5 packages with the longest names.
3. Write a function that runs a command with a timeout and retries it up to 3 times.

<details><summary>Show solutions</summary>

```python
import subprocess, sys, json

def git_branch():
    r = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None

pkgs = json.loads(subprocess.run([sys.executable, "-m", "pip", "list", "--format=json"],
                                 capture_output=True, text=True, check=True).stdout)
print(sorted((p["name"] for p in pkgs), key=len, reverse=True)[:5])

def run_with_retry(cmd, timeout=10, attempts=3):
    for i in range(attempts):
        try:
            return subprocess.run(cmd, capture_output=True, text=True, check=True, timeout=timeout)
        except (subprocess.TimeoutExpired, subprocess.CalledProcessError):
            if i == attempts - 1:
                raise
```
</details>